## BƯỚC 1: Khởi tạo feature engineering và các đường dẫn CSV.

In [1]:
# BƯỚC 1: Khởi tạo feature engineering và các đường dẫn CSV.
from pathlib import Path
import numpy as np
import pandas as pd

TARGET = "anomaly"  # Cột nhãn; sẽ được tách riêng khỏi ma trận feature.
MISSING_CATEGORY = "__MISSING__"  # Nhãn thay thế cho category bị thiếu.
UNKNOWN_CATEGORY = "__UNKNOWN__"  # Nhãn biểu diễn category không có trong train.

# Tìm data/ ngay cạnh notebook hoặc ở thư mục cha khi chạy từ thư mục con.
DATA_DIR = next(
    (path for path in (Path.cwd() / "data", Path.cwd().parent / "data") if path.is_dir()),
    None,
)
if DATA_DIR is None:  # Dừng sớm thay vì đọc nhầm đường dẫn hoặc báo lỗi ở bước sau.
    raise FileNotFoundError("Không tìm thấy thư mục data/.")

PROCESSED_DIR = DATA_DIR / "processed"  # Thư mục chứa CSV trung gian và kết quả.
TRAIN_INPUT = PROCESSED_DIR / "train_preprocessed.parquet"  # Train sau tiền xử lý.
TEST_INPUT = PROCESSED_DIR / "test_preprocessed.parquet"  # Test sau tiền xử lý.
TRAIN_OUTPUT = PROCESSED_DIR / "train_engineered.parquet"  # Train sau khi tạo/mã hóa feature.
TEST_OUTPUT = PROCESSED_DIR / "test_engineered.parquet"  # Test sau khi tạo/mã hóa feature.
STEP_LOG = []  # Danh sách ghi lại số dòng, giá trị và cột thay đổi ở từng bước.

# Kiểm tra phụ thuộc đầu vào để yêu cầu chạy notebook tiền xử lý trước nếu cần.
if not TRAIN_INPUT.exists() or not TEST_INPUT.exists():
    raise FileNotFoundError(
        "Chưa có CSV tiền xử lý; hãy chạy tuần tự preprocessingforALL.ipynb trước."
    )


def record_fe_step(step, train_before, train_after, test_before, test_after,
                   train_affected=0, test_affected=0, train_values_changed=0,
                   test_values_changed=0, columns_added=0, columns_removed=0):
    """Ghi kích thước hai tập và mức thay đổi của bước hiện tại."""
    entry = {
        "step": step,  # Tên thao tác để nhận biết từng dòng trong CSV log.
        "train_rows_before": train_before,  # Số mẫu train ngay trước bước.
        "train_rows_after": train_after,  # Số mẫu train ngay sau bước.
        "train_rows_added": max(train_after - train_before, 0),  # Số mẫu train được thêm.
        "train_rows_removed": max(train_before - train_after, 0),  # Số mẫu train bị bỏ.
        "train_rows_affected": train_affected,  # Số mẫu train có ít nhất một thay đổi.
        "train_values_changed": train_values_changed,  # Tổng ô/feature train được sửa/tạo.
        "test_rows_before": test_before,  # Số mẫu test ngay trước bước.
        "test_rows_after": test_after,  # Số mẫu test ngay sau bước.
        "test_rows_added": max(test_after - test_before, 0),  # Số mẫu test được thêm.
        "test_rows_removed": max(test_before - test_after, 0),  # Số mẫu test bị bỏ.
        "test_rows_affected": test_affected,  # Số mẫu test có ít nhất một thay đổi.
        "test_values_changed": test_values_changed,  # Tổng ô/feature test được sửa/tạo.
        "columns_added": columns_added,  # Số cột feature mới được tạo.
        "columns_removed": columns_removed,  # Số cột feature cũ bị loại.
    }
    STEP_LOG[:] = [previous for previous in STEP_LOG if previous["step"] != step]  # Thay log cũ cùng tên khi chạy lại ô.
    STEP_LOG.append(entry)  # Lưu thống kê của lần chạy hiện tại.
    from IPython.display import display, HTML
    col_change = entry.get('columns_added', 0) - entry.get('columns_removed', 0)
    col_str = f"<span style='color:green; font-weight:bold'>+{col_change}</span>" if col_change > 0 else (f"<span style='color:red; font-weight:bold'>{col_change}</span>" if col_change < 0 else "<span style='color:gray'>0</span>")
    
    html = (f"<b>Bước:</b> {step} | "
            f"<b>Train:</b> <span style='color:green; font-weight:bold'>+{entry['train_rows_added']}</span> / <span style='color:red; font-weight:bold'>-{entry['train_rows_removed']}</span> / <span style='color:blue; font-weight:bold'>~{entry['train_rows_affected']}</span> dòng | "
            f"<b>Test:</b> <span style='color:green; font-weight:bold'>+{entry['test_rows_added']}</span> / <span style='color:red; font-weight:bold'>-{entry['test_rows_removed']}</span> / <span style='color:blue; font-weight:bold'>~{entry['test_rows_affected']}</span> dòng | "
            f"<b>Cột:</b> {col_str}")
    display(HTML(html))

print(f"Nguồn train: {TRAIN_INPUT}")  # In file train sẽ được nạp ở bước kế tiếp.
print(f"Nguồn test:  {TEST_INPUT}")  # In file test sẽ được nạp ở bước kế tiếp.
print("Sẵn sàng đọc/ghi CSV bằng pandas.")

Nguồn train: /home/toon/p4aids/data/processed/train_preprocessed.parquet
Nguồn test:  /home/toon/p4aids/data/processed/test_preprocessed.parquet
Sẵn sàng đọc/ghi CSV bằng pandas.


## BƯỚC 2: Đọc các CSV đã được tiền xử lý.

In [2]:
# BƯỚC 2: Đọc các CSV đã được tiền xử lý.
train_fe_raw = pd.read_parquet(TRAIN_INPUT, engine="fastparquet")  # Nạp train và suy luận dtype nhất quán trên toàn file.
test_fe_raw = pd.read_parquet(TEST_INPUT, engine="fastparquet")  # Nạp test bằng cùng cách để schema nhất quán.
TRAIN_ROWS_INITIAL = len(train_fe_raw)  # Chốt số dòng train gốc để so sánh cuối pipeline.
TEST_ROWS_INITIAL = len(test_fe_raw)  # Chốt số dòng test gốc để so sánh cuối pipeline.

print(f"Train: {train_fe_raw.shape}")  # Báo số dòng/cột train vừa nạp.
print(f"Test:  {test_fe_raw.shape}")  # Báo số dòng/cột test vừa nạp.
print("Train dtypes:")  # Hiển thị nhãn kiểu dữ liệu để kiểm tra category/datetime.
print(train_fe_raw.dtypes.value_counts())  # Đếm số cột theo từng kiểu dữ liệu.

Train: (1749494, 53)
Test:  (1800567, 53)
Train dtypes:
float32    35
uint8      13
int64       4
int8        1
Name: count, dtype: int64


## BƯỚC 3: Tách target/row_id 

In [3]:
# BƯỚC 3: Tách target/row_id 
y_train = pd.to_numeric(train_fe_raw.pop(TARGET), errors="raise").reset_index(drop=True)  # Lấy nhãn khỏi train và ép về kiểu số.
test_row_ids = test_fe_raw.pop("row_id").reset_index(drop=True) if "row_id" in test_fe_raw else None  # Giữ ID để ghép dự đoán, không dùng làm feature.
X_train = train_fe_raw.copy()  # Sao chép phần còn lại của train thành ma trận feature.
X_test = test_fe_raw.copy()  # Sao chép phần còn lại của test thành ma trận feature.
assert X_train.columns.equals(X_test.columns), "Feature CSV train/test đang lệch cột."  # Dừng nếu schema khác nhau.

# Khai báo chu kỳ và offset dùng để kiểm tra cách tạo sin/cos cũ.
cycle_specs = {"hour": (24, 0), "weekday": (7, 0), "month": (12, 1)}  # Tháng bắt đầu từ 1; giờ/thứ từ 0.
cycle_audit = []  # Lưu sai số kiểm tra riêng cho từng chu kỳ.
for source, (period, offset) in cycle_specs.items():  # Kiểm tra lần lượt giờ, thứ và tháng.
    x_name, y_name = f"{source}_x", f"{source}_y"  # Tên hai cột cyclic cũ cần đối chiếu.
    if source not in X_train.columns or x_name not in X_train or y_name not in X_train:  # Bỏ qua nếu thiếu cột nguồn/cặp cũ.
        continue  # Không thể kiểm tra một chu kỳ chưa có đủ dữ liệu.
    sample_index = np.linspace(0, len(X_train) - 1, min(2000, len(X_train)), dtype=int)  # Chọn mẫu đều tối đa 2.000 dòng.
    source_values = pd.to_numeric(X_train[source].iloc[sample_index], errors="coerce").to_numpy()  # Lấy giá trị nguồn dạng số.
    angle = 2 * np.pi * ((source_values - offset) % period) / period  # Tính góc kỳ vọng theo chu kỳ và offset.
    expected_x, expected_y = np.cos(angle), np.sin(angle)  # Giả định x=cos, y=sin để kiểm định.
    actual_x = X_train[x_name].iloc[sample_index].to_numpy()  # Lấy cột x đã lưu trong file.
    actual_y = X_train[y_name].iloc[sample_index].to_numpy()  # Lấy cột y đã lưu trong file.
    max_error = float(max(np.nanmax(np.abs(actual_x - expected_x)),  # Đo sai số lớn nhất của cột x.
                          np.nanmax(np.abs(actual_y - expected_y))))  # Gộp sai số cột y vào cùng giá trị.
    cycle_audit.append({"source": source, "max_abs_error": max_error,  # Ghi tên nguồn và sai số để xem lại.
                        "valid_pair": bool(np.isclose(max_error, 0, atol=1e-5))})  # Đánh dấu cặp có khớp trong ngưỡng không.

record_fe_step("Tách target/row_id và audit chu kỳ", TRAIN_ROWS_INITIAL, len(X_train),  # Ghi số dòng và trạng thái bước.
               TEST_ROWS_INITIAL, len(X_test))  # Hoàn tất thông tin số dòng test.
print(f"X_train={X_train.shape}, y_train={y_train.shape}, X_test={X_test.shape}")  # Xác nhận kích thước X/y.
print(pd.DataFrame(cycle_audit).to_string(index=False))  # Hiển thị sai số cặp cyclic hiện có.

X_train=(1749494, 52), y_train=(1749494,), X_test=(1800567, 52)
Empty DataFrame
Columns: []
Index: []


## BƯỚC 4: Dùng Isolation Forest để tạo feature bất thường không giám sát.

In [4]:
# BƯỚC 4: Dùng Isolation Forest để tạo feature bất thường không giám sát.
# Forest chỉ fit trên feature train; nhãn anomaly không được đưa vào phép học.
import os  # Lấy số worker khả dụng để giới hạn tài nguyên cục bộ.
from sklearn.ensemble import IsolationForest  # Isolation Forest là tập hợp các isolation trees.

train_rows_before = len(X_train)  # Ghi row count train trước khi thêm score/flag.
test_rows_before = len(X_test)  # Ghi row count test trước khi thêm score/flag.
ISO_FIT_SAMPLE_SIZE = min(100_000, train_rows_before)  # Lấy tối đa 100.000 mẫu để fit nhanh và ổn định.
ISO_SCORE_BATCH_SIZE = 50_000  # Chấm theo batch để tránh tạo ma trận NumPy lớn trong RAM.
ISO_N_ESTIMATORS = 100  # Dùng 100 cây để tổng hợp nhiều phép cô lập ngẫu nhiên.
ISO_N_JOBS = max(1, min(os.cpu_count() or 1, 4))  # Giới hạn tối đa 4 worker cho laptop.

iso_fit_sample = X_train.sample(n=ISO_FIT_SAMPLE_SIZE, random_state=42)  # Lấy mẫu tái lập chỉ từ feature train.
iso_fit_array = iso_fit_sample.to_numpy(dtype=np.float32, copy=True)  # Chuyển mẫu sang float32 để giảm RAM khi fit.
isolation_model = IsolationForest(  # Khởi tạo tập cây cô lập.
    n_estimators=ISO_N_ESTIMATORS,  # Số cây độc lập trong ensemble.
    max_samples="auto",  # Mỗi cây dùng tối đa min(256, số mẫu fit) điểm.
    contamination="auto",  # Dùng ngưỡng mặc định của sklearn cho nhãn outlier.
    random_state=42,  # Cố định ngẫu nhiên để chạy lại cho cùng kết quả.
    n_jobs=ISO_N_JOBS,  # Cho phép song song nhưng giới hạn tài nguyên.
)
isolation_model.fit(iso_fit_array)  # Học cấu trúc bình thường/bất thường mà không dùng target.
del iso_fit_sample, iso_fit_array  # Giải phóng bản mẫu và mảng dùng riêng cho fit.


def score_with_isolation_forest(frame):  # Tính score/flag cho toàn bộ một tập theo batch.
    scores = np.empty(len(frame), dtype=np.float32)  # Cấp mảng score đầu ra gọn trong RAM.
    flags = np.empty(len(frame), dtype=np.int8)  # Cấp mảng nhị phân 1=bất thường, 0=bình thường.
    for start in range(0, len(frame), ISO_SCORE_BATCH_SIZE):  # Duyệt các lát dữ liệu có kích thước giới hạn.
        stop = min(start + ISO_SCORE_BATCH_SIZE, len(frame))  # Chặn batch cuối không vượt số dòng.
        batch = frame.iloc[start:stop].to_numpy(dtype=np.float32, copy=True)  # Chỉ chuyển batch hiện tại sang NumPy.
        batch_scores = isolation_model.decision_function(batch)  # Score thấp hơn nghĩa là điểm cô lập hơn.
        scores[start:stop] = batch_scores  # Ghi score về đúng vị trí dòng gốc.
        flags[start:stop] = (batch_scores < 0).astype("int8")  # Theo ngưỡng auto, score âm được gắn cờ outlier.
        del batch, batch_scores  # Giải phóng bộ nhớ tạm trước batch kế tiếp.
    return scores, flags  # Trả hai feature có cùng thứ tự dòng với đầu vào.

train_isolation_scores, train_isolation_flags = score_with_isolation_forest(X_train)  # Chấm toàn bộ train bằng model fit từ train sample.
test_isolation_scores, test_isolation_flags = score_with_isolation_forest(X_test)  # Chấm test bằng chính model đã fit, không fit lại.
X_train["isolation_score"] = train_isolation_scores  # Thêm score liên tục; càng thấp càng bất thường.
X_test["isolation_score"] = test_isolation_scores  # Thêm cùng feature cho test.
X_train["isolation_flag"] = train_isolation_flags  # Thêm cờ outlier 0/1 cho train.
X_test["isolation_flag"] = test_isolation_flags  # Thêm cùng cờ cho test.

record_fe_step("Isolation Forest score/flag", train_rows_before, len(X_train),  # Ghi thay đổi số dòng train.
               test_rows_before, len(X_test),  # Ghi thay đổi số dòng test.
               train_affected=train_rows_before, test_affected=test_rows_before,  # Mỗi dòng nhận score và flag.
               train_values_changed=2 * train_rows_before,  # Hai feature Isolation Forest trên mỗi dòng train.
               test_values_changed=2 * test_rows_before,  # Hai feature Isolation Forest trên mỗi dòng test.
               columns_added=2)  # Thêm isolation_score và isolation_flag.
train_flag_rate = float(X_train["isolation_flag"].mean())  # Tính tỷ lệ flag train để so sánh với test.
test_flag_rate = float(X_test["isolation_flag"].mean())  # Tính tỷ lệ flag test để phát hiện lệch phân phối.
print(f"Fit mẫu train: {ISO_FIT_SAMPLE_SIZE:,} | số cây: {ISO_N_ESTIMATORS} | workers: {ISO_N_JOBS}")  # Báo cấu hình model.
print(f"Train bị gắn cờ: {int(X_train['isolation_flag'].sum()):,}/{len(X_train):,} ({train_flag_rate:.1%})")  # Hiển thị tỷ lệ train.
print(f"Test bị gắn cờ: {int(X_test['isolation_flag'].sum()):,}/{len(X_test):,} ({test_flag_rate:.1%})")  # Hiển thị tỷ lệ test.
print("Train score quantiles:")  # Tóm tắt phân phối score train.
print(X_train["isolation_score"].quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 1]))  # Phân vị giúp đọc mức độ bất thường.
print("Test score quantiles:")  # Tóm tắt phân phối score test để soi data drift.
print(X_test["isolation_score"].quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 1]))  # So với train; score thấp hơn nghĩa là lệch khỏi train.
if test_flag_rate > max(0.5, train_flag_rate * 3):  # Cảnh báo khi test bị flag cao bất thường so với train.
    print("CẢNH BÁO: flag test cao; đây là dấu hiệu distribution shift, không phải nhãn anomaly thật.")  # Nhắc cần kiểm tra dữ liệu/encoding.
del train_isolation_scores, train_isolation_flags, test_isolation_scores, test_isolation_flags  # Giải phóng mảng sau khi gắn vào X.

Fit mẫu train: 100,000 | số cây: 100 | workers: 4
Train bị gắn cờ: 276,306/1,749,494 (15.8%)
Test bị gắn cờ: 231,613/1,800,567 (12.9%)
Train score quantiles:
0.00   -0.119282
0.01   -0.040975
0.10   -0.007765
0.50    0.030393
0.90    0.062577
0.99    0.077968
1.00    0.096726
Name: isolation_score, dtype: float64
Test score quantiles:
0.00   -0.096101
0.01   -0.031774
0.10   -0.003938
0.50    0.030923
0.90    0.062332
0.99    0.078168
1.00    0.096726
Name: isolation_score, dtype: float64


## BƯỚC 5: Ghi Parquet feature engineered và các bảng nhật ký.

In [5]:
# BƯỚC 5: Ghi Parquet feature engineered và các bảng nhật ký.
train_output = X_train.copy(deep=False)  # Tạo view output để không nhân bản ma trận train lớn.
train_output[TARGET] = y_train.to_numpy()  # Gắn nhãn vào file train, không đưa nhãn vào X_train.
test_output = X_test.copy(deep=False)  # Tạo view output test để giảm mức dùng RAM.
if test_row_ids is not None:  # Chỉ thêm ID nếu input test có row_id.
    test_output.insert(0, "row_id", test_row_ids.to_numpy())  # Giữ ID đầu bảng để ghép dự đoán sau này.

train_output.to_parquet(TRAIN_OUTPUT, index=False, engine="fastparquet")  # Ghi train thành Parquet, không lưu index pandas.
test_output.to_parquet(TEST_OUTPUT, index=False, engine="fastparquet")  # Ghi test theo cùng cấu hình.
train_saved_rows = len(pd.read_parquet(TRAIN_OUTPUT, engine="fastparquet", columns=[TARGET]))  # Đếm lại Parquet train theo lô.
test_saved_rows = len(pd.read_parquet(TEST_OUTPUT, engine="fastparquet", columns=[test_output.columns[0]]))  # Đếm lại Parquet test theo lô.
assert train_saved_rows == len(X_train), "Sai số dòng train Parquet."
assert test_saved_rows == len(X_test), "Sai số dòng test Parquet."

record_fe_step("Xuất Parquet engineered (không biến đổi)", len(X_train), len(X_train),  # Ghi bước lưu file, không đổi dữ liệu.
               len(X_test), len(X_test))  # Ghi row count test trước/sau lưu.
STEP_LOG_TABLE = pd.DataFrame(STEP_LOG)  # Đổi nhật ký các bước thành bảng để hiển thị/lưu.
STEP_LOG_PATH = PROCESSED_DIR / "feature_engineering_step_log.parquet"  # Đường dẫn log thay đổi.
STEP_LOG_TABLE.to_parquet(STEP_LOG_PATH, index=False, engine="fastparquet")  # Xuất log pipeline thành Parquet, không ghi index.

print(f"Train: {TRAIN_OUTPUT} | {train_saved_rows:,} dòng | "
      f"{TRAIN_OUTPUT.stat().st_size / 1024**2:.1f} MB")  # Báo file train, row count và dung lượng.
print(f"Test:  {TEST_OUTPUT} | {test_saved_rows:,} dòng | "
      f"{TEST_OUTPUT.stat().st_size / 1024**2:.1f} MB")  # Báo file test, row count và dung lượng.
print(f"Nhật ký bước: {STEP_LOG_PATH}")  # Cho biết nơi lưu số liệu thay đổi.
# Hiển thị log rút gọn có màu sắc
from IPython.display import HTML, display

summary_log = STEP_LOG_TABLE[['step']].copy()
summary_log.rename(columns={'step': 'Bước xử lý'}, inplace=True)

def format_row(added, removed, affected):
    return f"<span style='color:green; font-weight:bold'>+{added}</span> / " \
           f"<span style='color:red; font-weight:bold'>-{removed}</span> / " \
           f"<span style='color:blue; font-weight:bold'>~{affected}</span>"

summary_log['Train (Thêm / Xóa / Chỉnh)'] = STEP_LOG_TABLE.apply(
    lambda r: format_row(r['train_rows_added'], r['train_rows_removed'], r['train_rows_affected']), axis=1
)
summary_log['Test (Thêm / Xóa / Chỉnh)'] = STEP_LOG_TABLE.apply(
    lambda r: format_row(r['test_rows_added'], r['test_rows_removed'], r['test_rows_affected']), axis=1
)

if 'columns_added' in STEP_LOG_TABLE.columns:
    summary_log['Biến động Cột'] = STEP_LOG_TABLE['columns_added'].apply(
        lambda x: f"<span style='color:green; font-weight:bold'>+{x}</span>" if x > 0 else 
                  (f"<span style='color:red; font-weight:bold'>{x}</span>" if x < 0 else 
                   "<span style='color:gray'>0</span>")
    )

html_table = summary_log.to_html(escape=False, index=False)
display(HTML(html_table))

Train: /home/toon/p4aids/data/processed/train_engineered.parquet | 1,749,494 dòng | 39.2 MB
Test:  /home/toon/p4aids/data/processed/test_engineered.parquet | 1,800,567 dòng | 46.1 MB
Nhật ký bước: /home/toon/p4aids/data/processed/feature_engineering_step_log.parquet


Bước xử lý,Train (Thêm / Xóa / Chỉnh),Test (Thêm / Xóa / Chỉnh),Biến động Cột
Tách target/row_id và audit chu kỳ,+0 / -0 / ~0,+0 / -0 / ~0,0
Isolation Forest score/flag,+0 / -0 / ~1749494,+0 / -0 / ~1800567,+2
Xuất Parquet engineered (không biến đổi),+0 / -0 / ~0,+0 / -0 / ~0,0


## BƯỚC 6: Tổng kết Feature Engineering
Chỉ hiển thị danh sách các biến mới được sinh ra ở quá trình Feature Engineering.

In [6]:
# BƯỚC 6: Tổng kết các Feature cuối cùng (Chỉ hiển thị các biến mới sinh ra ở phần Feature Engineering)
import pandas as pd

column_details = {
    'isolation_flag': ('Định tính', 'Cờ (0/1) báo hiệu đây có phải là điểm bất thường dưới góc nhìn của Isolation Forest.'),
    'isolation_score': ('Định lượng', 'Điểm số bất thường (anomaly score) do thuật toán Isolation Forest đánh giá độc lập.')
}

new_fe_columns = ['isolation_score', 'isolation_flag']
new_fe_columns = [col for col in new_fe_columns if col in X_train.columns] # Đảm bảo cột tồn tại

print('='*60)
print(f'CÁC BIẾN MỚI THỰC SỰ ĐƯỢC TẠO RA Ở FE (Tổng số: {len(new_fe_columns)} cột mới)')
print('='*60)
summary_df = pd.DataFrame({
    'Tên Cột Mới': new_fe_columns,
    'Loại Biến': [column_details.get(col, ('Khác', ''))[0] for col in new_fe_columns],
    'Ý Nghĩa / Phương pháp xử lý': [column_details.get(col, ('', ''))[1] for col in new_fe_columns]
})

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 150)
display(summary_df)


CÁC BIẾN MỚI THỰC SỰ ĐƯỢC TẠO RA Ở FE (Tổng số: 2 cột mới)


,Tên Cột Mới,Loại Biến,Ý Nghĩa / Phương pháp xử lý
0,isolation_score,Định lượng,Điểm số bất thường (anomaly score) do thuật toán Isolation Forest đánh giá độc lập.
1,isolation_flag,Định tính,Cờ (0/1) báo hiệu đây có phải là điểm bất thường dưới góc nhìn của Isolation Forest.
